# Group C synthesis: Strands from the inside and from labels

**Question.** Along one ladder, how far does a single decision model (Strands 2B) get from text when it is
read more deeply (its own probabilities → its hidden `<answer>` vector, both label-free) and when it is
trained on the labels (head only → LoRA only → LoRA + head), compared with methods that use labels by design
(TAR, the label-trained classifier)?

This notebook brings C1 (`C/C1`) and C2 (`C/C2`) together without redrawing
their panels: one ladder on the joint question, raw gain per task family, with the measured training GPU hours
of each trained cell, and the same ladder on the MulTaBench-style normalised score that pools ROC-AUC and R². Each experiment keeps its own comparisons and statistics. It reads only the two
verified Group C exports.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.notebook_display import display_json, display_local, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import group_c, group_c_views, study_export
from src.visualize import figures, group_c as gcfig, style
style.apply()
NOTEBOOK = "C/C_synthesis"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
display_json("Notebook execution identity", provenance())
c1, c1_receipt = group_c.read("C1")
c2, c2_receipt = group_c.read("C2")
for study, receipt in (("C1", c1_receipt), ("C2", c2_receipt)):
    display_local(f"{study}: {study_export.describe(receipt)}")
    record({k: receipt.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "schema")}, f"{study} export identity")
ladder = group_c_views.overview(group_c_views.combined_gains(c1, c2))
gpu_hours = group_c_views.training_gpu_hours(c2["fits"]) if c2 else {}

'C1: Waiting for export: C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\C\\C1\\results\\group_c\\c1_representations.zip is absent. No measurements are shown.'

'C2: Waiting for export: C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\C\\C2\\results\\group_c\\c2_adaptation.zip is absent. No measurements are shown.'

## 1. What is complete

The ladder needs the C2 export (zero-shot, trained cells, TAR and classifier) and the C1 export (the hidden
vector). A rung absent from the exports is left out of the figure and named here.

In [2]:
report_section("1. What is complete")
present = set(ladder.representation) if not ladder.empty else set()
missing = [gcfig.label(rep) for rep in group_c_views.LADDER if rep not in present]
display_local(f"C1 export: {c1_receipt['status']}; C2 export: {c2_receipt['status']}.")
display_local("Ladder rungs waiting for an export: " + ("; ".join(missing) if missing else "none."))

'C1 export: awaiting_export; C2 export: awaiting_export.'

'Ladder rungs waiting for an export: Strands probabilities, joint; <answer>, layer 24, joint; Head only; LoRA only; LoRA + head; TAR E5-small, per column PCA32; TAR E5-small, joint PCA64; Label-trained classifier, E5-small'

## 2. The Strands ladder

From zero-shot probabilities to the label-free hidden vector, then to the three trained cells, beside TAR and
the label-trained classifier; raw gain over no text per task family (predictor mean with dataset-bootstrap
95% interval, dataset median and each predictor), with total training GPU hours in the right margin. Below it, the same rungs and the no-text baseline on the MulTaBench-style normalised score (per dataset and fold, the headline score min-max normalised across every predictor × rung configuration, negative R² clipped at 0, all-tied folds skipped; equal-dataset mean and median over all datasets). It is relative to this comparison set and complements the per-family panels.

In [3]:
report_section("2. The Strands ladder")
if ladder.empty:
    display_local("Waiting for the Group C exports: nothing to show yet.")
else:
    gcfig.render(save, gcfig.strands_ladder(ladder, name="strands_ladder", gpu_hours=gpu_hours))
    rungs = ladder[ladder.representation.isin(group_c_views.LADDER) & ladder.model_id.eq(group_c_views.PREDICTOR_MEAN)]
    display_table("Ladder: mean over predictors per task family (equal-dataset mean, median and 95% interval)", rungs[["representation", "task_family", "datasets", "mean_raw_gain", "raw_ci_low", "raw_ci_high", "median_raw_gain"]])
    record(gpu_hours, "Training GPU hours per C2 cell (sum over fits)")
    _, normalized = group_c_views.normalized_ladder(c1, c2)
    gcfig.render(save, gcfig.strands_normalized_ladder(normalized, name="strands_ladder_normalized_score"))
    display_table("Ladder on the normalised score: per rung and predictor, all datasets and per task family", normalized, preview_rows=20)

'Waiting for the Group C exports: nothing to show yet.'

## 3. Complete report

In [4]:
finish_report(NOTEBOOK, save.summary() + "\nOnly verified Group C exports were read; no inference, training or VSC change was performed.")

C/C_synthesis

Notebook execution identity
{
  "code_sha256": "253d5f17bdea4be35b0e853055326f9a06bd5d0a5b2b0c789f83d236fd3c5074",
  "git_head": "0929454751c9921410a46b29506eb6c2b4ca280f",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}

C1: Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\C\C1\results\group_c\c1_representations.zip is absent. No measurements are shown.


C1 export identity
{
  "status": "awaiting_export",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\C\\C1\\results\\group_c\\c1_representations